# 5 · Distributed computing on FABRIC — a gate between two QPUs

The other concept notebooks spend entanglement on a **key**. This one spends it on a
**gate**: alice and bob hold no qubit in common, so no two-qubit gate between them is
possible unless they consume a Bell pair and put correction bits on the wire.

Runs `node_runner --protocol dqc` across the alice and bob slice nodes. Alice is the
control node and hosts the register; bob is the target node and reaches his qubits by
RPC. m1 crosses A→B inside `PLAN`, m2 returns B→A inside `ACK`, and **each side applies
the value as it arrived** — which is what makes the dropped-bit control below real
rather than simulated.

Three things this notebook shows, in order:

1. a non-local CNOT is exact over the link when the pair is perfect;
2. a noisy pair costs `(1−w)/2` — *the same curve as the E91 QBER*, so one number
   prices a distributed gate and a distributed key;
3. a **permanently lost** correction costs 50% on the one Pauli channel it protects,
   while a **late** one is recovered exactly — so herald latency is a memory-time
   cost, not a fidelity cost.

Concepts: `docs/PRIMER.md` §4.4 and `docs/CONCEPTS.md` §6.1.
Prereq: run `fabric/01_setup_slice` first.

## 1 · Configuration

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'          # same slice as fabric/01
SCENARIO   = 'validation/scenarios/fabric_1km.yml'

PRIMITIVE  = 'telegate'   # 'telegate' (non-local CNOT) | 'teleport' (state transfer)
NUM_GATES  = 2000         # per Pauli channel; a run spends 2x this many Bell pairs
CLASSICAL  = 'tcp'        # 'tcp' | 'l2' (raw 0x7102 through the BMv2 switch)

# Werner weights to sweep for the error law. 1.0 = a perfect pair.
WEIGHTS    = [1.0, 0.95, 0.9, 0.8]

## 2 · Load the slice

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## 3 · Ship code + build runtime

In [ ]:
df.upload_project(slice_obj)
df.setup_sequence_runtime(slice_obj)
print('runtime ready on alice + bob')

## 4 · One exact gate over the link

A perfect pair (w = 1) and an intact classical channel. Both Pauli channels should read
**zero**: the non-local CNOT is exact across two nodes that share no qubit.

In [ ]:
a_exact, b_exact = df.run_sequence_dqc(
    slice_obj, primitive=PRIMITIVE, num_gates=NUM_GATES,
    fidelity=1.0, classical_transport=CLASSICAL)

print(f"\nbit_error   = {a_exact['bit_error']}")
print(f"phase_error = {a_exact['phase_error']}")
print(f"pairs spent = {a_exact['pairs_consumed']}, classical bits = {a_exact['classical_bits']}")
print(f"both sides agree: {a_exact['bit_error'] == b_exact['bit_error']}")

The phase channel is the part worth pausing on. It prepares the control in |+⟩ and the
target in |0⟩, so a *correct* CNOT leaves the two nodes holding |Φ⁺⟩ — an entangled state
they **computed**, out of entanglement the network supplied. Reading it out in X and
finding the outcomes agree is what rules out a classical copy of a measured bit.

## 5 · The price of a noisy pair

Sweep the Werner weight. Both channels should track `(1−w)/2` — the identical curve the
E91 QBER follows, which is the platform's point: pair quality is one number that prices
both services on a link.

In [ ]:
rows = []
for w in WEIGHTS:
    a, _ = df.run_sequence_dqc(slice_obj, primitive=PRIMITIVE, num_gates=NUM_GATES,
                               fidelity=w, classical_transport=CLASSICAL)
    rows.append((w, a['bit_error'], a['phase_error'], a['error_pred']))
    print(f"  w={w}: bit={a['bit_error']:.4f} phase={a['phase_error']:.4f} "
          f"predicted={a['error_pred']:.4f}")

print(f"\n{'w':>6} {'bit':>9} {'phase':>9} {'(1-w)/2':>9}")
for w, b, ph, pred in rows:
    print(f"{w:>6} {b:>9.4f} {ph:>9.4f} {pred:>9.4f}")

In [ ]:
ws  = [r[0] for r in rows]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ws, [(1 - w) / 2 for w in ws], 'k--', label='law  (1−w)/2  = E91 QBER')
ax.plot(ws, [r[1] for r in rows], 'o-', label='bit error (X-type)')
ax.plot(ws, [r[2] for r in rows], 's-', label='phase error (Z-type)')
ax.set_xlabel('Werner weight w of the distributed pair')
ax.set_ylabel('error rate per distributed gate')
ax.set_title(f'{PRIMITIVE} on FABRIC: a gate priced like a key')
ax.invert_xaxis(); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6 · The classical channel is load-bearing — but latency is not the enemy

Two controls on a **perfect** pair, so every error below comes from the wire alone.

* `dropped` — the correction never arrives. Each bit protects exactly one Pauli channel,
  so losing it costs 50% there and nothing on the other.
* `late` — the correction misses its gate and is folded into the recorded outcome
  afterwards (a tracked Pauli frame, the same move the repeater makes when it
  XOR-composes heralds). The result comes back **clean**.

That distinction is the finding: a late Pauli correction is recoverable, so herald
latency is charged against *memory coherence*, not fidelity. An error appears only when
a correction is permanently lost, or lands after the result was consumed.

In [ ]:
controls = {}
for label, kw in [('clean',      {}),
                  ('drop m1',    {'dropped': 'm1'}),
                  ('drop m2',    {'dropped': 'm2'}),
                  ('late m1',    {'late': 'm1'}),
                  ('late m2',    {'late': 'm2'})]:
    a, _ = df.run_sequence_dqc(slice_obj, primitive=PRIMITIVE, num_gates=NUM_GATES // 2,
                               fidelity=1.0, classical_transport=CLASSICAL, **kw)
    controls[label] = (a['bit_error'], a['phase_error'])

print(f"{'run':>10} {'bit':>9} {'phase':>9}")
for k, (b, ph) in controls.items():
    print(f"{k:>10} {b:>9.4f} {ph:>9.4f}")

## 7 · Verify

In [ ]:
# Which Pauli channel each correction bit protects. This is NOT symmetric between
# the two primitives, and hard-coding telegate's mapping would fail a perfectly
# correct teleport run: telegate sends m1 -> X on bob's target and m2 -> Z on
# alice's control, while a teleport correction is X^m2.Z^m1 -- so there m2 is the
# X (bit channel) and m1 the Z (phase channel).
PROTECTS = {'telegate': {'m1': 'bit', 'm2': 'phase'},
            'teleport': {'m1': 'phase', 'm2': 'bit'}}[PRIMITIVE]
CHANNEL = {'bit': 0, 'phase': 1}

def drop_breaks_only_its_channel(bit):
    hit = CHANNEL[PROTECTS[bit]]
    spared = 1 - hit
    r = controls[f'drop {bit}']
    return r[hit] > 0.4 and r[spared] < 0.05

checks = []
checks.append(('exact gate at w=1',
               a_exact['bit_error'] == 0.0 and a_exact['phase_error'] == 0.0))
checks.append(('both nodes report the same run',
               a_exact['gates_completed'] == b_exact['gates_completed']))
checks.append(('error follows (1-w)/2 on both channels',
               all(abs(b - pred) < 0.02 and abs(ph - pred) < 0.02
                   for _, b, ph, pred in rows)))
checks.append((f'a dropped bit breaks exactly the channel it protects ({PRIMITIVE})',
               drop_breaks_only_its_channel('m1') and drop_breaks_only_its_channel('m2')))
checks.append(('a late bit is recovered exactly',
               controls['late m1'] == (0.0, 0.0) and controls['late m2'] == (0.0, 0.0)))

for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(ok for _, ok in checks), 'some checks failed -- see the per-run output above'

## Takeaways

* Two QPUs that share **no qubit** ran a CNOT between them, over a real link, by spending
  one Bell pair and one classical bit each way. The control never left its node — which is
  why a distributed compiler emits telegates rather than teleports.
* The phase channel produced |Φ⁺⟩ **as the output of a computation**, correlated in Z and
  in X, from entanglement the network delivered.
* A distributed gate and a distributed key degrade on the **same curve**, `(1−w)/2`.
* Losing a correction bit is a Pauli error you cannot resend your way out of; a *late*
  one costs only the memory time the qubit must survive. At WAN distances that wait, not
  the fiber, is what limits distributed computation — the fidelity model here does not
  yet decay with hold time (`docs/ASSUMPTIONS.md`), so treat these numbers as a floor.